# H2S-NN Graph Classification - Colab

Colab runner for TU graph classification. Default protocol is 5 stratified holdout runs with train/validation/test = 0.8/0.1/0.1, validation AUROC checkpointing, and patience-100 early stopping.

## 1. Mount Drive And Enter Repo

Edit `PROJECT_DIR` if your repo lives somewhere else.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)
print('Python executable:', sys.executable)


## 2. Install Dependencies

This keeps the active Colab PyTorch build and installs PyG wheels for that Torch/CUDA pair.

In [ ]:
import subprocess
import sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cpu' if torch.version.cuda is None else 'cu' + torch.version.cuda.replace('.', '')
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('torch:', torch.__version__)
print('pyg wheel url:', wheel_url)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scikit-learn', 'pandas', 'matplotlib', 'tqdm', 'wandb', 'umap-learn', 'optuna'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', wheel_url])


## 3. Sanity Check

In [ ]:
import subprocess
import sys
import torch
import torch_geometric

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch_geometric:', torch_geometric.__version__)
subprocess.run([sys.executable, 'scripts/train_h2snn.py', '--help'], check=True)
subprocess.run([sys.executable, 'scripts/run_h2snn_runs.py', '--help'], check=True)
subprocess.run([sys.executable, 'scripts/run_h2snn_wandb_tuning.py', '--help'], check=True)


## 4. Choose Dataset And Write Colab Config

The graph readout is fixed (sum of `[x_K, h0]` per graph -> BatchNorm -> MLP); ablations go in the config's `ablation:` block.

In [ ]:
from pathlib import Path
import yaml

DATASET = 'proteins'       # one of: 'mutag', 'proteins', 'enzymes', 'nci1'
NUM_RUNS = 5
BASE_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
EPOCHS = 200
MONITOR = 'auroc'
EARLY_STOPPING_PATIENCE = 100
USE_WANDB = False
WANDB_PROJECT = 'h2snn'
WANDB_MODE = 'online'

CONFIG_BY_DATASET = {
    'mutag': 'configs/experiments/mutag_h2snn.yaml',
    'proteins': 'configs/experiments/proteins_h2snn.yaml',
    'enzymes': 'configs/experiments/enzymes_h2snn.yaml',
    'nci1': 'configs/experiments/nci1_h2snn.yaml',
}

base_config = Path(CONFIG_BY_DATASET[DATASET])
payload = yaml.safe_load(base_config.read_text(encoding='utf-8'))
exp_name = f'{DATASET}_h2snn_colab'
payload['epochs'] = EPOCHS
payload['seed'] = BASE_SEED
payload['device'] = DEVICE
payload['exp_name'] = exp_name
payload['output_dir'] = 'experiments/colab'
payload['checkpoint_metric'] = MONITOR
payload['checkpoint_mode'] = 'max'
payload['early_stopping_patience'] = EARLY_STOPPING_PATIENCE
payload['use_wandb'] = USE_WANDB
payload['wandb_project'] = WANDB_PROJECT
payload['wandb_mode'] = WANDB_MODE
payload.setdefault('data', {})['split_protocol'] = 'stratified_holdout'
payload['data']['num_runs'] = NUM_RUNS
payload['data']['run_index'] = 0
payload['data']['train_ratio'] = TRAIN_RATIO
payload['data']['val_ratio'] = VAL_RATIO
payload['data']['test_ratio'] = TEST_RATIO

CONFIG_PATH = Path('configs/generated/colab') / f'{exp_name}.yaml'
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')

EXP_NAME = exp_name
print('base config:', base_config)
print('colab config:', CONFIG_PATH)
print('experiment:', EXP_NAME)


## 5. Optional W&B Login

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()
else:
    print('W&B disabled. Set USE_WANDB = True in the config cell to enable tracking.')


## 6. Run 5 Independent Seeds

In [ ]:
import os
import subprocess
import sys

command = [
    sys.executable,
    '-u',
    'scripts/run_h2snn_runs.py',
    '--config',
    str(CONFIG_PATH),
    '--device',
    DEVICE,
    '--exp_name',
    EXP_NAME,
    '--num_runs',
    str(NUM_RUNS),
    '--base_seed',
    str(BASE_SEED),
    '--monitor',
    MONITOR,
    '--patience',
    str(EARLY_STOPPING_PATIENCE),
        '--train_ratio',
        str(TRAIN_RATIO),
        '--val_ratio',
        str(VAL_RATIO),
        '--test_ratio',
        str(TEST_RATIO),
]
if USE_WANDB:
    command.extend(['--use_wandb', '--wandb_project', WANDB_PROJECT, '--wandb_mode', WANDB_MODE])

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Training failed with exit code {return_code}')


## 7. Inspect 5-Run Report

In [ ]:
from pathlib import Path
import json

summary_path = Path('experiments/colab') / EXP_NAME / 'runs_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing summary: {summary_path}')

payload = json.loads(summary_path.read_text(encoding='utf-8'))
metrics = ['test_auroc', 'test_macro_f1', 'test_micro_f1', 'test_acc']
for record in payload['runs']:
    pieces = [f"run {record.get('run_index')} seed={record.get('seed')} best_epoch={record.get('best_epoch', record.get('epoch'))}"]
    pieces.extend(f"{metric}={float(record.get(metric, 0.0)):.4f}" for metric in metrics)
    print(' | '.join(pieces))
print('')
for metric, values in payload['summary']['metrics'].items():
    print(f"{metric}: mean={values['mean']:.4f} std={values['std']:.4f}")
print('summary:', summary_path)


## 8. Optional W&B Bayesian + Hyperband Tuning

This creates one W&B sweep per seed/run. Each seed selects its own best trial by validation metric, then `tuning_summary.json` reports mean/std over the five selected run scores.

In [ ]:
RUN_WANDB_TUNING = False
TUNE_TRIALS_PER_SEED = 20
TUNE_METRIC = 'auroc'
INCLUDE_DECODER_TYPE_IN_TUNING = False
print('W&B tuning enabled:', RUN_WANDB_TUNING)
print('Tune metric:', TUNE_METRIC)


In [ ]:
import os
import subprocess
import sys

if RUN_WANDB_TUNING:
    command = [
        sys.executable,
        '-u',
        'scripts/run_h2snn_wandb_tuning.py',
        '--config',
        str(CONFIG_PATH),
        '--device',
        DEVICE,
        '--exp_name',
        f'{EXP_NAME}_tune',
        '--num_runs',
        str(NUM_RUNS),
        '--base_seed',
        str(BASE_SEED),
        '--n_trials',
        str(TUNE_TRIALS_PER_SEED),
        '--monitor',
        TUNE_METRIC,
        '--patience',
        str(EARLY_STOPPING_PATIENCE),
        '--wandb_project',
        WANDB_PROJECT,
        '--wandb_mode',
        WANDB_MODE,
    ]
    if 'DECODER_TYPE' in globals():
        command.extend(['--decoder_type', DECODER_TYPE])
    if INCLUDE_DECODER_TYPE_IN_TUNING:
        command.append('--include_decoder_type')

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('Running:', ' '.join(command))
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'W&B tuning failed with exit code {return_code}')
else:
    print('Skipping W&B tuning. Normal 5-run reporting is in the training section.')
